# Simple block-MAP decoding of log(numerosity) from the mapper

**Goal**: show that the fitted Gaussian numerosity pRFs in NPCr carry enough information to decode the stimulus numerosity on a per-block basis.

**Trick** (to avoid HRF-convolved likelihood code): we do NOT try to decode at TR resolution against an HRF-convolved forward model. Instead:

1. Load the already-fitted per-voxel pRF parameters (`mu`, `sd`, `amplitude`, `baseline`, `r2`) from `derivatives/encoding_model/`.
2. Load the run-mean cleaned BOLD timeseries `desc-meancleaned_space-T1w_bold.nii.gz` (written by `fit_mapper.py`).
3. For each stimulus block, extract the mean BOLD activity in the window `[onset + 4s, onset + duration + 4s]` — a crude HRF-peak window.
4. For each block, over a log-numerosity grid, compute $-\sum_{v \in \text{NPC}_r, \text{top-}N} (a_v(\log n) - y_v)^2$ and pick the MAP log n.

Because both encoding parameters and block activities come from the same (run-averaged) data, this is **in-sample** and biased upward. It's a sanity check that the fitted tuning curves align with the actual stimulus-locked activity, not a decoding benchmark.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
import seaborn as sns
from nilearn import image

BIDS = Path('~/data/ds-balgrist').expanduser()
SESSIONS = ['balgrist3t', 'balgrist7t', 'ibt7t', 'sns3t']
HRF_LAG_S = 4.0      # window start relative to block onset
HRF_TAIL_S = 4.0     # window end relative to block offset
TOP_N_VOX = 100      # number of NPCr voxels ranked by training R²

In [ ]:
def load_pars(subject, session, bids=BIDS):
    base = bids / 'derivatives' / 'encoding_model' / f'sub-{subject}' / f'ses-{session}' / 'func'
    names = ['mu', 'sd', 'amplitude', 'baseline', 'r2']
    imgs = {k: nib.load(str(base / f'sub-{subject}_ses-{session}_task-mapper_desc-{k}_space-T1w_pars.nii.gz'))
            for k in names}
    return imgs


def load_meancleaned(subject, session, bids=BIDS):
    fn = (bids / 'derivatives' / 'encoding_model' / f'sub-{subject}' / f'ses-{session}' / 'func'
          / f'sub-{subject}_ses-{session}_task-mapper_desc-meancleaned_space-T1w_bold.nii.gz')
    return nib.load(str(fn))


def load_npc_mask(subject, roi='NPCr', bids=BIDS):
    return nib.load(str(bids / 'derivatives' / 'masks' / f'sub-{subject}' / 'anat'
                        / f'sub-{subject}_space-T1w_desc-{roi}_mask.nii.gz'))


def load_events(subject, session, run, bids=BIDS):
    fn = bids / f'sub-{subject}' / f'ses-{session}' / 'func' \
         / f'sub-{subject}_ses-{session}_task-mapper_run-{run}_events.tsv'
    return pd.read_csv(fn, sep='\t')


def get_tr(subject, session, run=1, bids=BIDS):
    fn = bids / f'sub-{subject}' / f'ses-{session}' / 'func' \
         / f'sub-{subject}_ses-{session}_task-mapper_run-{run}_bold.json'
    with open(fn) as f:
        return float(json.load(f)['RepetitionTime'])

## One subject / session at a time — inspect the pipeline first

In [ ]:
subject = '01'
session = 'balgrist3t'

pars = load_pars(subject, session)
mean_bold = load_meancleaned(subject, session)
npcr = load_npc_mask(subject, 'NPCr')
tr = get_tr(subject, session)

# Intersect NPCr with voxels that were actually fit (R² ≠ 0 → inside fit mask)
npcr_arr = npcr.get_fdata() > 0
r2_arr = pars['r2'].get_fdata()
fitmask = (r2_arr != 0)
sel_mask = npcr_arr & fitmask
print(f'NPCr voxels total:   {int(npcr_arr.sum()):,}')
print(f'NPCr ∩ fit mask:     {int(sel_mask.sum()):,}')

# Pull per-voxel pars and activity within NPCr∩fitmask
idx = np.where(sel_mask)
mu   = pars['mu'].get_fdata()[idx]
sd   = pars['sd'].get_fdata()[idx]
amp  = pars['amplitude'].get_fdata()[idx]
base = pars['baseline'].get_fdata()[idx]
r2   = r2_arr[idx]
bold = mean_bold.get_fdata()[idx]            # (n_vox, n_vols)
n_vox, n_vols = bold.shape
print(f'n_vox={n_vox}  n_vols={n_vols}  TR={tr}')

In [ ]:
# Top-N voxels by R²
top_n = min(TOP_N_VOX, n_vox)
order = np.argsort(r2)[::-1][:top_n]
print(f'using top {top_n} voxels  (R² range: {r2[order].min():.3f}–{r2[order].max():.3f})')

mu_sel, sd_sel = mu[order], sd[order]
amp_sel, base_sel = amp[order], base[order]
bold_sel = bold[order]                       # (top_n, n_vols)

# Quick look at a few tuning curves
log_n_grid = np.linspace(np.log(3), np.log(120), 80)
fig, ax = plt.subplots(figsize=(6, 4))
for i in np.random.default_rng(0).choice(top_n, size=min(10, top_n), replace=False):
    y = amp_sel[i] * np.exp(-((log_n_grid - mu_sel[i])**2) / (2 * sd_sel[i]**2)) + base_sel[i]
    ax.plot(np.exp(log_n_grid), y, alpha=0.7)
ax.set_xscale('log')
ax.set_xlabel('n dots')
ax.set_ylabel('predicted % signal change')
ax.set_title(f'10 random NPCr pRFs — sub-{subject} ses-{session}')
plt.tight_layout()

## Extract per-block mean activity, then MAP-decode

In [ ]:
def block_mean_activity(bold, events, tr, lag=HRF_LAG_S, tail=HRF_TAIL_S):
    """For each stimulation block, mean BOLD over [onset+lag, onset+duration+tail].
    Returns (n_blocks, n_vox) activity + a DataFrame with the block metadata.
    """
    stim = events[events['trial_type'] == 'stimulation'].copy()
    frame_times = np.arange(bold.shape[1]) * tr
    rows = []
    acts = []
    for _, ev in stim.iterrows():
        t0 = float(ev['onset']) + lag
        t1 = float(ev['onset']) + float(ev['duration']) + tail
        mask = (frame_times >= t0) & (frame_times <= t1)
        if mask.sum() < 1:
            continue
        acts.append(bold[:, mask].mean(axis=1))
        rows.append({'onset': float(ev['onset']),
                     'n_dots': float(ev['n_dots']),
                     'log_n': float(np.log(ev['n_dots'])),
                     'n_tr_in_window': int(mask.sum())})
    return np.stack(acts, axis=0), pd.DataFrame(rows)

# The meancleaned BOLD is the *average* over runs at each volume — so block onsets from run-1
# correspond to volumes that were averaged with the matching block from the other runs.
# This works because the mapper presents the same stimulus ordering on every run.
events_run1 = load_events(subject, session, run=1)

block_acts, block_meta = block_mean_activity(bold_sel, events_run1, tr)
print(f'{len(block_meta)} stimulus blocks,  activity shape {block_acts.shape}')
block_meta.head()

In [ ]:
def map_decode(block_acts, mu, sd, amp, base, stim_grid):
    """Return (n_blocks,) MAP log-numerosity, minimising SSE over the grid."""
    # tuning[i, v, g] is not computed — instead loop over grid for memory:
    n_blocks, n_vox = block_acts.shape
    sse = np.zeros((n_blocks, len(stim_grid)), dtype=np.float64)
    for gi, log_n in enumerate(stim_grid):
        pred = amp * np.exp(-((log_n - mu) ** 2) / (2 * sd ** 2)) + base   # (n_vox,)
        sse[:, gi] = ((block_acts - pred[None, :]) ** 2).sum(axis=1)
    return stim_grid[sse.argmin(axis=1)], sse

decoded, sse = map_decode(block_acts, mu_sel, sd_sel, amp_sel, base_sel, log_n_grid)

df_blocks = block_meta.copy()
df_blocks['decoded_log_n'] = decoded
df_blocks['decoded_n'] = np.exp(decoded)
df_blocks.head(12)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
sns.stripplot(data=df_blocks, x='n_dots', y='decoded_n',
              order=sorted(df_blocks['n_dots'].unique()),
              jitter=0.15, alpha=0.6, ax=ax)
ax.plot([1, 100], [1, 100], 'k--', alpha=0.4)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('true n dots')
ax.set_ylabel('decoded n (MAP)')
ax.set_title(f'sub-{subject} ses-{session} — NPCr top-{top_n}  (in-sample)')
from scipy.stats import spearmanr
rho, _ = spearmanr(df_blocks['log_n'], df_blocks['decoded_log_n'])
print(f'Spearman rho (log): {rho:.3f}')
plt.tight_layout()

## Sweep all (subject, session) combinations

In [ ]:
from scipy.stats import spearmanr

def decode_session(subject, session, top_n=TOP_N_VOX, bids=BIDS):
    pars = load_pars(subject, session, bids)
    mean_bold = load_meancleaned(subject, session, bids)
    try:
        npcr = load_npc_mask(subject, 'NPCr', bids)
    except FileNotFoundError:
        return None
    tr = get_tr(subject, session, bids=bids)

    npcr_arr = npcr.get_fdata() > 0
    fitmask = pars['r2'].get_fdata() != 0
    sel_mask = npcr_arr & fitmask
    if sel_mask.sum() < 10:
        return None
    idx = np.where(sel_mask)
    mu, sd = pars['mu'].get_fdata()[idx], pars['sd'].get_fdata()[idx]
    amp, base = pars['amplitude'].get_fdata()[idx], pars['baseline'].get_fdata()[idx]
    r2 = pars['r2'].get_fdata()[idx]
    bold = mean_bold.get_fdata()[idx]

    n_available = len(r2)
    tn = min(top_n, n_available)
    order = np.argsort(r2)[::-1][:tn]
    mu_s, sd_s, amp_s, base_s = mu[order], sd[order], amp[order], base[order]
    bold_s = bold[order]

    events = load_events(subject, session, run=1, bids=bids)
    block_acts, block_meta = block_mean_activity(bold_s, events, tr)
    log_n_grid = np.linspace(np.log(3), np.log(120), 80)
    decoded, _ = map_decode(block_acts, mu_s, sd_s, amp_s, base_s, log_n_grid)
    block_meta['decoded_log_n'] = decoded
    block_meta['subject'] = subject
    block_meta['session'] = session
    block_meta['field_strength'] = '7T' if '7t' in session else '3T'
    block_meta['n_vox_used'] = tn
    block_meta['r2_min_used'] = float(r2[order].min())
    return block_meta


subjects = sorted({p.name.split('sub-')[1]
                   for p in (BIDS / 'derivatives' / 'encoding_model').iterdir()
                   if p.is_dir() and p.name.startswith('sub-')})
results = []
for sub in subjects:
    for ses in SESSIONS:
        try:
            res = decode_session(sub, ses)
            if res is not None:
                results.append(res)
                rho, _ = spearmanr(res['log_n'], res['decoded_log_n'])
                print(f'sub-{sub}  ses-{ses}  n_blocks={len(res)}  n_vox={res.n_vox_used.iloc[0]}  rho={rho:+.3f}')
        except (FileNotFoundError, KeyError) as e:
            print(f'sub-{sub}  ses-{ses}  SKIP: {e}')
all_blocks = pd.concat(results, ignore_index=True)
all_blocks.head()

In [ ]:
perf = (all_blocks.groupby(['subject', 'session', 'field_strength'])
                   .apply(lambda g: spearmanr(g['log_n'], g['decoded_log_n'])[0])
                   .reset_index(name='rho'))
print(perf.pivot(index='subject', columns='session', values='rho').round(3))

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.barplot(data=perf, x='session', y='rho', hue='field_strength',
            order=['balgrist3t', 'sns3t', 'balgrist7t', 'ibt7t'], ax=ax)
ax.axhline(0, color='k', lw=0.5)
ax.set_ylabel('Spearman ρ (true vs decoded log n)')
ax.set_title(f'In-sample block-MAP decoding in NPCr (top-{TOP_N_VOX} voxels)')
plt.tight_layout()

In [ ]:
# Scatter grid: decoded vs true per (subject × session)
subjects = sorted(all_blocks['subject'].unique())
sessions = ['balgrist3t', 'sns3t', 'balgrist7t', 'ibt7t']
fig, axes = plt.subplots(len(subjects), len(sessions),
                         figsize=(3.2 * len(sessions), 3 * len(subjects)),
                         sharex=True, sharey=True)
for i, sub in enumerate(subjects):
    for j, ses in enumerate(sessions):
        ax = axes[i, j] if len(subjects) > 1 else axes[j]
        g = all_blocks[(all_blocks['subject'] == sub) & (all_blocks['session'] == ses)]
        if g.empty:
            ax.set_axis_off()
            continue
        sns.stripplot(data=g, x='n_dots', y='decoded_log_n',
                      order=sorted(g['n_dots'].unique()),
                      jitter=0.15, alpha=0.6, ax=ax, size=3)
        # Reference diagonal
        xs = sorted(g['n_dots'].unique())
        ax.plot(range(len(xs)), np.log(xs), 'k--', alpha=0.4)
        rho, _ = spearmanr(g['log_n'], g['decoded_log_n'])
        ax.set_title(f'sub-{sub}  ses-{ses}\nρ={rho:+.2f}', fontsize=9)
        ax.tick_params(axis='x', rotation=45)
        if i == len(subjects) - 1:
            ax.set_xlabel('true n dots')
        if j == 0:
            ax.set_ylabel('decoded log n')
plt.tight_layout()